[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/icsl-aist/hsr-genesis/blob/main/examples/tutorials/3_arm_control_colab.ipynb)


# チュートリアル 3: FK/IK を用いたアーム制御 / Tutorial 3: Arm control using FK/IK

## 目的 / Objective

**JP:** このノートブックでは、HSR のアームを制御する方法を学びます。
順運動学（FK）で手先位置を計算し、名前付き姿勢・直接関節制御・逆運動学（IK）を試します。

**EN:** In this notebook you will learn how to control the HSR's arm.
We will compute hand positions with forward kinematics (FK), then try
named poses, direct joint control, and inverse kinematics (IK).

> **Note:** Run this on a GPU runtime (Colab: *Runtime → Change runtime type → T4 GPU* or better).

## Setup / セットアップ

Run the cell below to install dependencies, clone the repo, and configure GPU rendering.
If you run into issues, see the [troubleshoot notebook](7_troubleshoot_colab.ipynb).

下のセルを実行して、依存パッケージのインストール、リポジトリのクローン、GPU レンダリングの設定を行います。
問題が発生した場合は[トラブルシューティングノートブック](7_troubleshoot_colab.ipynb)を参照してください。


In [ ]:
import importlib, urllib.request

# Fetch standalone bootstrap from GitHub (zero hsr_genesis imports).
exec(urllib.request.urlopen(
    "https://raw.githubusercontent.com/icsl-aist/hsr-genesis/main/colab_setup.py"
).read())

# One-call setup: installs deps, clones repo (with submodules),
# configures EGL for headless GPU rendering. Safe to re-run.
# See the troubleshoot notebook if anything goes wrong.
setup_colab()


In [ ]:
from hsr_genesis.tutorial_utils import *

init_sim()


## 5. 順運動学（FK） / Forward kinematics

**JP:** 順運動学（FK）は、関節角度が与えられたときに手先（`hand_palm_link`）の
位置と姿勢を計算します。`forward_kinematics(arm_angles)` は各リンクの
4×4 同次変換行列を返します。

関節の順序は `[arm_lift, arm_flex, arm_roll, wrist_flex, wrist_roll]`（単位: rad）です。

**EN:** Forward kinematics (FK) computes the position and orientation of the
hand (`hand_palm_link`) given joint angles. `forward_kinematics(arm_angles)`
returns a dict of 4×4 homogeneous transforms for each link.

The joint order is `[arm_lift, arm_flex, arm_roll, wrist_flex, wrist_roll]` (in radians).

In [ ]:
# 順運動学で手先位置を計算 / Compute hand position via forward kinematics
# arm_angles = [arm_lift, arm_flex, arm_roll, wrist_flex, wrist_roll] [rad]
fk_result = forward_kinematics([0.0, -0.8, 0.0, -0.4, 0.0])

# hand_palm_link の位置を表示 / Print the hand_palm_link position
hand_T = fk_result['hand_palm_link']
print('hand_palm_link position (x, y, z):', hand_T[:3, 3])

## 6. 名前付き姿勢 / Named poses

**JP:** `tutorial_utils` には便利な名前付き姿勢が用意されています:

- `move_arm_neutral()` — ニュートラル（準備）姿勢
- `move_arm_init()` — 初期（ホーム）姿勢

どちらも `duration` [s] を返すので `run()` に渡せます。

**EN:** `tutorial_utils` provides convenient named poses:

- `move_arm_neutral()` — neutral (ready) pose
- `move_arm_init()` — initial (home) pose

Both return the `duration` [s] which you can pass to `run()`.

In [ ]:
# ニュートラル姿勢へ移動 / Move to the neutral pose
move_arm_neutral()
run(2.0)
show_video()

In [ ]:
# 初期姿勢へ移動 / Move to the init (home) pose
move_arm_init()
run(2.0)
show_video()

## 7. 直接関節制御 / Direct joint control

**JP:** `move_arm_joints(angles)` で 5 つのアーム関節を直接指定できます。
`angles` は `[arm_lift, arm_flex, arm_roll, wrist_flex, wrist_roll]`（単位: rad）のリストです。

**EN:** `move_arm_joints(angles)` lets you directly specify the 5 arm joints.
`angles` is a list of `[arm_lift, arm_flex, arm_roll, wrist_flex, wrist_roll]` in radians.

In [ ]:
# 関節を直接指定して移動 / Move arm joints directly
# [arm_lift, arm_flex, arm_roll, wrist_flex, wrist_roll] [rad]
move_arm_joints([0.2, -1.2, 0.5, -0.6, 0.3])
run(2.0)
show_video()

## 8. 逆運動学（IK） / Inverse kinematics

**JP:** 逆運動学（IK）は、手先の目標位置と姿勢が与えられたときに、
それを実現する関節角度を計算します。
`move_wholebody_ik(x, y, z, roll, pitch, yaw)` は全身 IK を用いて
ベース位置とアーム関節の両方を最適化し、目標姿勢に到達します。

- `x, y, z`: 手先の目標位置 [m]
- `roll, pitch, yaw`: 手先の目標姿勢 [deg]
- `duration`: 目標到達までの時間 [s]（戻り値としても返されます）

**EN:** Inverse kinematics (IK) computes the joint angles needed to reach
a target hand position and orientation.
`move_wholebody_ik(x, y, z, roll, pitch, yaw)` uses whole-body IK to
optimize both the base position and arm joints to reach the target.

- `x, y, z`: Target end-effector position [m]
- `roll, pitch, yaw`: Target end-effector orientation [deg]
- `duration`: Time to reach the target [s] (also returned as the return value)

In [ ]:
# IK で手先を目標位置へ移動 / Move the hand to a target pose via IK
# target: x=0.5, y=0.2, z=0.3 [m], roll=0, pitch=90, yaw=0 [deg]
move_wholebody_ik(0.5, 0.2, 0.3, 0, 90, 0)
run(3.0)
show_video()

## 9. 手先位置の確認 / Check the hand position

**JP:** `get_hand_pos()` は現在の手先（`hand_palm_link`）位置を `(x, y, z)` として返します。
IK の目標に近い値になっているか確認しましょう。

**EN:** `get_hand_pos()` returns the current hand (`hand_palm_link`) position
as `(x, y, z)`. Let's verify it is close to the IK target.

In [ ]:
# 手先位置を確認 / Check current hand position
print(get_hand_pos())

## 10. ニュートラル姿勢への復帰 / Return to the neutral pose

**JP:** 最後にアームをニュートラル姿勢に戻します。

**EN:** Finally, return the arm to the neutral pose.

In [ ]:
# ニュートラル姿勢へ復帰 / Return to the neutral pose
move_arm_neutral()
run(2.0)
show_video()

## まとめ / Summary

**JP:** このチュートリアルでは以下を学びました:

1. `forward_kinematics(arm_angles)` — 順運動学で手先位置を計算
2. `move_arm_neutral()` / `move_arm_init()` — 名前付き姿勢への移動
3. `move_arm_joints(angles)` — 直接関節制御
4. `move_wholebody_ik(x, y, z, roll, pitch, yaw)` — 逆運動学による手先制御
5. `get_hand_pos()` — 手先位置の取得

**EN:** In this tutorial you learned:

1. `forward_kinematics(arm_angles)` — computing hand positions via FK
2. `move_arm_neutral()` / `move_arm_init()` — moving to named poses
3. `move_arm_joints(angles)` — direct joint control
4. `move_wholebody_ik(x, y, z, roll, pitch, yaw)` — hand control via IK
5. `get_hand_pos()` — reading the hand position

これで HSR のベース制御とアーム制御の基本を習得しました。
次は把持のチュートリアルに進みましょう。
You have now mastered the basics of HSR base and arm control.
Next, proceed to the grasping tutorial.